# Draw Keyjoint

## PyTorch

In [ ]:
import torch
import json
import os
import pandas as pd
import numpy as np
import torchvision
from torchvision.utils import draw_keypoints, make_grid, draw_bounding_boxes
from PIL import Image
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=True)
model.eval()

In [ ]:
# refer online
COCO_SKELETON_0_INDEXED = [
    (1, 0), (2, 0),
    (3, 1), (4, 2),
    (5, 6),
    (5, 7), (6, 8),
    (7, 9), (8, 10),
    (5, 11), (6, 12),
    (11, 13), (12, 14),
    (13, 15), (14, 16)
]
parts = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]
column_names = []
for part in parts:
  column_names.extend([f"{part}_x", f"{part}_y"])
keypoint_data = []

In [ ]:
with open('/content/sitting-posture-v2-coco/train/_annotations.coco.json', 'r') as f:
    coco_data = json.load(f)

In [ ]:
def get_keypoints(image_id):
  #COCO JSON
  img_info = next(item for item in coco_data['images'] if item['id'] == image_id)
  file_name = img_info['file_name']

  ann_info = [item for item in coco_data['annotations'] if item['image_id'] == image_id]

  image_path = os.path.join('/content/sitting-posture-v2-coco/train', file_name)

  img = Image.open(image_path).convert("RGB")
  img_tensor = torch.as_tensor(np.array(img)).permute(2, 0, 1)

  bbox = []
  cat_id = 0
  for ann in ann_info:
    x,y,w,h = ann['bbox']
    cat_id = ann['category_id']
    bbox.append([x,y,x+w,y+h])

  bbox_tensor = torch.tensor(bbox, dtype=torch.float32)
  img_with_boxes = draw_bounding_boxes(img_tensor, bbox_tensor, colors='red',width=2)
  img_for_drawing = F.to_tensor(img).unsqueeze(0).to(device)

  with torch.no_grad():
      output = model(img_for_drawing)[0]

  keypoints = output['keypoints']
  keypoints_scores = output['keypoints_scores'] # score for the coord

  #Skip if empty result
  if (keypoints.shape[0] == 0) : return
  scores = output['scores']

  # 90% con7firm result
  high_score_kp = keypoints[scores > 0.9]
  high_score_kp_scores = keypoints_scores[scores > 0.9]
  #debug
  print(scores)
  print(high_score_kp)

  if high_score_kp.shape[0] == 0: return

  #if more than one
  selected_kp = high_score_kp[0]
  selected_kp_scores = high_score_kp_scores[0]
  print(selected_kp_scores)

  kp_filtered = selected_kp.clone()

  for i in range(17):
    if selected_kp_scores[i] < 0.0:
        kp_filtered[i, :] = torch.tensor([-1, -1, 0])

  #store csv
  data_row = kp_filtered[:, :2].reshape(-1).cpu().numpy()
  df = pd.DataFrame([data_row], columns=column_names)
  df.insert(0, 'image_id', image_id)
  df['cat_id'] = cat_id
  keypoint_data.append(df)

  #draw kp on image

  if len(selected_kp[0]) > 0:
    pred_keypoints, visibility = kp_filtered.split([2, 1], dim=-1)
    visibility = visibility.bool()

    annotated_img = draw_keypoints(
        img_with_boxes,
        pred_keypoints.unsqueeze(0),
        visibility=visibility.unsqueeze(0),
        colors='red',
        radius=4,
        connectivity=COCO_SKELETON_0_INDEXED
    )

  #just tengok
  grid = make_grid(annotated_img)
  grid_np = grid.permute(1, 2, 0).numpy() #(H,W,C)

  plt.figure(figsize=(10, 10))
  plt.imshow(grid_np)
  plt.title("Torch!")
  plt.axis('off')
  plt.show()

  output_filename = "keyjoint_train_dataset/keyjoints_" + os.path.basename(image_path)
  torchvision.io.write_png((annotated_img).to(torch.uint8), output_filename)

In [ ]:
import os

# Create the output directory if it doesn't exist
output_dir = "keyjoint_train_dataset"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

for image_info in coco_data['images']:
    image_id = image_info['id']
    print(f"Processing image ID: {image_id}")
    get_keypoints(image_id=image_id)

In [ ]:
get_keypoints(image_id=2)

In [ ]:
combined_keypoint_df = pd.concat(keypoint_data, ignore_index=True)
combined_keypoint_df.to_csv("keypoint_train_data.csv", index=False)

In [ ]:
!zip -r /content/keyjoint_train_dataset.zip /content/keyjoint_train_dataset